# Stage 2: Investigate answer-letter bias

Run the following cell in the existing Stage 2 Colab session, after its baseline has finished. It uses the same model and a fixed development subset. It preserves the original results, rotates answer choices, reverses display order, and tests one explicit answer prefix. It does not choose a replacement prompt or evaluate the final test split.


In [ ]:
# Stage 2 diagnostic: preserve the original baseline and examine answer-letter bias.
import os
import subprocess
import sys
import zipfile
from pathlib import Path
from google.colab import files, userdata

os.chdir('/content/unlearning_task')
if not Path('outputs/stage2/full/run.json').exists():
    raise RuntimeError('Restore the original Stage 2 results before running this diagnostic.')
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()

diagnostic_source = '"""Fixed, development-only checks for option-position and answer-format bias.\n\nThis does not select a new prompt, change the baseline, or tune on final-test data.\n"""\n\nimport json\nfrom pathlib import Path\nimport statistics\n\nfrom .baseline import development_examples, load_evaluator, read_predictions\nfrom .data import digest, file_digest, write_json\nfrom .evaluation import LABELS, MCQEvaluator, next_token_logits, padded_inputs, summarize_logits\nfrom .runtime import environment_info\n\nVARIANTS = ("original", "assistant_prefix", "reversed_labels")\nSANITY = [\n    {"id": "sanity-even", "question": "Which number is even?", "choices": ["3", "4", "5", "7"], "answer": 1},\n    {"id": "sanity-count", "question": "A box contains two red balls and one blue ball. How many balls are there in total?",\n     "choices": ["one", "two", "three", "four"], "answer": 2},\n    {"id": "sanity-capital", "question": "What is the capital of France?",\n     "choices": ["Rome", "Berlin", "Madrid", "Paris"], "answer": 3},\n    {"id": "sanity-color", "question": "On a clear sunny day, what color does the sky usually appear?",\n     "choices": ["blue", "purple", "green", "orange"], "answer": 0},\n]\n\n\ndef select_examples(examples, seed):\n    """16 forget + two per retain subject; never select by prediction or correctness."""\n    selected = []\n    subjects = sorted({row["subject"] for row in examples if row["role"] == "retain"})\n    for role, subject, count in [("forget", None, 16)] + [("retain", name, 2) for name in subjects]:\n        pool = [row for row in examples if row["role"] == role and (subject is None or row["subject"] == subject)]\n        if len(pool) < count:\n            raise ValueError("Insufficient development questions for the fixed diagnostic sample.")\n        pool.sort(key=lambda row: digest([seed, "bias-diagnostic-v1", row["id"]]))\n        selected.extend(pool[:count])\n    return selected\n\n\ndef rotate_example(example, shift):\n    """Map each displayed A-D slot back to its original choice index."""\n    order = [(i + shift) % 4 for i in range(4)]\n    return dict(example, choices=[example["choices"][i] for i in order],\n                answer=order.index(example["answer"])), order\n\n\ndef diagnostic_prompt(tokenizer, example, settings, variant):\n    if variant not in VARIANTS:\n        raise ValueError("Unknown diagnostic condition.")\n    pairs = list(zip(LABELS, example["choices"]))\n    if variant == "reversed_labels":\n        pairs.reverse()  # Same label-content mapping, but D is physically first.\n    text = settings["instruction"] + "\\n\\n" + example["question"] + "\\n"\n    text += "\\n".join("{}. {}".format(label, choice) for label, choice in pairs)\n    prompt = tokenizer.apply_chat_template([{"role": "user", "content": text}], tokenize=False,\n        add_generation_prompt=True, date_string=settings["date_string"])\n    candidates = list(LABELS)\n    if variant == "assistant_prefix":\n        prompt += "The correct answer is"\n        candidates = [" " + label for label in LABELS]\n    ids = tokenizer.encode(prompt, add_special_tokens=False)\n    if not ids or len(ids) > settings["max_input_tokens"]:\n        raise ValueError("Diagnostic prompt is too long; no truncation permitted.")\n    label_ids = []\n    for suffix in candidates:\n        whole = tokenizer.encode(prompt + suffix, add_special_tokens=False)\n        if whole[:-1] != ids or len(whole) != len(ids) + 1:\n            raise ValueError("Diagnostic answer suffix is not one token at the actual prompt boundary.")\n        label_ids.append(whole[-1])\n    if len(set(label_ids)) != 4:\n        raise ValueError("Diagnostic answer tokens are not distinct.")\n    return {"prompt": prompt, "prompt_hash": digest(prompt), "input_ids": ids,\n            "candidate_suffixes": candidates, "label_ids": label_ids}\n\n\nclass DiagnosticEvaluator(MCQEvaluator):\n    def score_condition(self, example, variant):\n        import torch\n        prepared = diagnostic_prompt(self.tokenizer, example, self.settings, variant)\n        inputs = padded_inputs([prepared], self.pad_id, self.device)\n        with torch.inference_mode():\n            full = next_token_logits(self.model, inputs)[0]\n            selected = full[prepared["label_ids"]]\n            mass = float((selected.logsumexp(-1) - full.logsumexp(-1)).exp())\n            top = int(full.argmax())\n        return dict(summarize_logits(selected.cpu().tolist(), example["answer"]),\n                    **prepared, answer_probability_mass=mass,\n                    unconstrained_next_token_id=top,\n                    unconstrained_next_token=self.tokenizer.decode([top]))\n\n    def direct_reference(self, example):\n        """No optimized last-logit path or explicit position IDs: all-position reference."""\n        import torch\n        prepared = diagnostic_prompt(self.tokenizer, example, self.settings, "original")\n        ids = torch.tensor([prepared["input_ids"]], device=self.device)\n        with torch.inference_mode():\n            logits = self.model(input_ids=ids, use_cache=False, return_dict=True).logits[0, -1].float()\n        return summarize_logits(logits[prepared["label_ids"]].cpu().tolist(), example["answer"])\n\n    def harmless_continuation(self, example, variant):\n        """At most 12 greedy tokens on invented harmless questions only."""\n        import torch\n        if not example["id"].startswith("sanity-"):\n            raise ValueError("Free-text continuation is restricted to the harmless sanity questions.")\n        prepared = diagnostic_prompt(self.tokenizer, example, self.settings, variant)\n        ids = list(prepared["input_ids"])\n        generated = []\n        stops = getattr(self.model.generation_config, "eos_token_id", None)\n        stops = set(stops if isinstance(stops, list) else [stops])\n        with torch.inference_mode():\n            for _ in range(12):\n                inputs = padded_inputs([{"input_ids": ids}], self.pad_id, self.device)\n                token = int(next_token_logits(self.model, inputs)[0].argmax())\n                generated.append(token)\n                if token in stops:\n                    break\n                ids.append(token)\n        return {"id": example["id"], "variant": variant, "generated_token_ids": generated,\n                "continuation": self.tokenizer.decode(generated, skip_special_tokens=True),\n                "note": "Greedy continuation, capped at 12 tokens; may be incomplete. Not the baseline scoring rule."}\n\n\ndef condition_summary(rows):\n    summary = {}\n    for variant in VARIANTS:\n        summary[variant] = {}\n        for role in ("forget", "retain", "sanity"):\n            group = [r for r in rows if r["variant"] == variant and r["role"] == role]\n            ids = sorted({r["id"] for r in group})\n            if not group:\n                continue\n            stable = sum(len({r["original_choice_prediction"] for r in group if r["id"] == identity}) == 1 for identity in ids)\n            same_label = sum(len({r["prediction"] for r in group if r["id"] == identity}) == 1 for identity in ids)\n            unrotated = [r for r in group if r["shift"] == 0]\n            summary[variant][role] = {\n                "unique_questions": len(ids), "evaluations": len(group),\n                "unrotated_accuracy": sum(r["correct"] for r in unrotated) / len(unrotated),\n                "rotation_averaged_accuracy": sum(r["correct"] for r in group) / len(group),\n                "content_consistency_across_rotations": stable / len(ids),\n                "same_label_across_rotations": same_label / len(ids),\n                "predicted_labels": {label: sum(r["prediction"] == label for r in group) for label in LABELS},\n                "mean_answer_probability_mass": statistics.mean(r["answer_probability_mass"] for r in group),\n                "note": "Four orderings of each question are correlated, not four independent samples."}\n    return summary\n\n\ndef run_diagnostic(data_dir="data/prepared/full", baseline_dir="outputs/stage2/full",\n                   output="outputs/stage2_diagnostic", evaluator=None):\n    baseline_dir, output = Path(baseline_dir), Path(output)\n    original_run = json.loads((baseline_dir / "run.json").read_text(encoding="utf-8"))\n    original_rows = {row["id"]: row for row in read_predictions(baseline_dir, original_run)}\n    context = original_run["context"]\n    if len(original_rows) != len(context["examples"]):\n        raise ValueError("Complete original baseline evidence is required.")\n    config, settings = context["config"], context["settings"]\n    manifest, development = development_examples(data_dir, config)\n    if manifest["request_hash"] != context["data_request_hash"]:\n        raise ValueError("Diagnostic data differs from the original baseline.")\n    selected = select_examples(development, config["seed"])\n    selected += [dict(row, role="sanity", subject="invented", content_hash=digest(row)) for row in SANITY]\n    base = evaluator if evaluator is not None else load_evaluator(config, settings)\n    probe = DiagnosticEvaluator(base.model, base.tokenizer, settings)\n    protocol = {"schema_version": 1, "baseline_fingerprint": original_run["fingerprint"],\n                "config": config, "settings": settings, "variants": list(VARIANTS),\n                "shifts": [0, 1, 2, 3], "ids": [row["id"] for row in selected],\n                "environment": environment_info(), "source_hash": file_digest(Path(__file__)),\n                "sample_rule": "hash-selected, 16 forget and 2 per retain subject; independent of original predictions",\n                "purpose": "Diagnosis only. No automatic prompt selection, model switch, or final-test evaluation."}\n    fingerprint = digest(protocol)\n    output.mkdir(parents=True, exist_ok=True)\n    protocol_path = output / "protocol.json"\n    if protocol_path.exists():\n        if json.loads(protocol_path.read_text(encoding="utf-8")) != protocol:\n            raise ValueError("Diagnostic protocol changed; use a new output directory.")\n    elif any(output.iterdir()):\n        raise ValueError("Diagnostic output has files without a protocol.")\n    else:\n        write_json(protocol_path, protocol)\n    rows, reference_checks, continuations = [], [], []\n    expected_count = len(selected) * len(VARIANTS) * 4\n    for example in selected:\n        for variant in VARIANTS:\n            for shift in range(4):\n                transformed, order = rotate_example(example, shift)\n                path = output / "predictions" / (digest([example["id"], variant, shift]) + ".json")\n                if path.exists():\n                    row = json.loads(path.read_text(encoding="utf-8"))\n                    checksum = row.pop("record_hash")\n                    if checksum != digest(row) or row["protocol_hash"] != fingerprint:\n                        raise ValueError("Diagnostic checkpoint mismatch.")\n                    row["record_hash"] = checksum\n                else:\n                    result = probe.score_condition(transformed, variant)\n                    row = dict(result, id=example["id"], role=example["role"], subject=example["subject"],\n                               variant=variant, shift=shift, choice_order=order, protocol_hash=fingerprint,\n                               original_choice_prediction=order[LABELS.index(result["prediction"])])\n                    row["record_hash"] = digest(row)\n                    write_json(path, row)\n                rows.append(row)\n                if len(rows) % 48 == 0:\n                    print("Diagnostic {}/{} checks saved.".format(len(rows), expected_count), flush=True)\n                if variant == "original" and shift == 0 and example["role"] != "sanity":\n                    old = original_rows[example["id"]]\n                    if row["input_ids"] != old["input_ids"] or row["prompt"] != old["prompt"]:\n                        raise ValueError("Original prompt replay does not match the saved baseline.")\n                    error = max(abs(row["answer_log_probabilities"][l] - old["answer_log_probabilities"][l]) for l in LABELS)\n                    reference_checks.append({"id": example["id"], "kind": "saved_baseline_replay",\n                                             "max_log_probability_error": error,\n                                             "passed": error <= settings["score_atol"]})\n        # Compare against an all-position native forward on two real questions per role.\n        prior = sum(r["kind"] == "native_forward" and r["role"] == example["role"] for r in reference_checks)\n        if example["role"] != "sanity" and prior < 2:\n            reference = probe.direct_reference(example)\n            result = next(r for r in rows if r["id"] == example["id"] and r["variant"] == "original" and r["shift"] == 0)\n            error = max(abs(result["answer_log_probabilities"][l] - reference["answer_log_probabilities"][l]) for l in LABELS)\n            reference_checks.append({"id": example["id"], "kind": "native_forward", "role": example["role"],\n                                     "max_log_probability_error": error, "passed": error <= settings["score_atol"]})\n        if example["role"] == "sanity":\n            for variant in ("original", "assistant_prefix"):\n                continuations.append(probe.harmless_continuation(example, variant))\n    summary = {"status": "complete", "protocol_hash": fingerprint, "final_test_evaluated": False,\n               "conditions": condition_summary(rows), "reference_checks": reference_checks,\n               "numerical_checks_passed": all(row["passed"] for row in reference_checks),\n               "harmless_continuations": continuations,\n               "next_action": "Review replay, label-versus-position behavior, and prompt sensitivity. Do not automatically pick the highest accuracy or replace the baseline."}\n    write_json(output / "summary.json", summary)\n    return summary\n\n\nif __name__ == "__main__":\n    print(json.dumps(run_diagnostic(), indent=2))\n'
target = Path('src/unlearning/diagnostics.py')
if target.exists() and target.read_text(encoding='utf-8') != diagnostic_source:
    raise RuntimeError('A different diagnostic version already exists; review before replacing it.')
target.write_text(diagnostic_source, encoding='utf-8')
del diagnostic_source
result = subprocess.run([sys.executable, '-m', 'unlearning.diagnostics'])
archive_path = Path('/content/stage2-diagnostic-results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(Path('outputs/stage2_diagnostic').rglob('*.json')):
        archive.write(path, path.as_posix())
    archive.write(target, target.as_posix())
files.download(str(archive_path))
if result.returncode:
    raise RuntimeError('Diagnostic stopped. The available evidence was downloaded for review.')
import json
report = json.loads(Path('outputs/stage2_diagnostic/summary.json').read_text())
print(json.dumps(report, indent=2))
